# MLB Phase 1D — In-game 모델 재검증 및 재학습

**2025 정규시즌 2,430경기**

이 노트북은 원고의 In-game 8변수를 유지하면서 **반이닝 시작 직전(pre-half-inning)** 스냅샷을 구축합니다. 따라서 완료된 해당 반이닝의 득점/안타 및 최종 경기 상태를 사전에 알았던 것처럼 사용하지 않습니다.

- 입력: **`phase1c_output.zip`** (게임별 train/validation/test 분할 포함)
- 필수 조건: 동일 경기의 관측치가 서로 다른 분할에 들어가지 않음. 2025년 3~8월 학습/검증, 9월 테스트 구조 유지
- 검사: API의 실제 진행된 반이닝을 `allPlays`에서 확인하고, `linescore.innings`에서 **이전 반이닝까지의 집계**만 입력으로 구성
- 모델: Keras DNN(8→64→32→16→1) + Logistic Regression + 이닝/점수차 기반 경험적 승률표 + 단순 앞선 팀 규칙
- 지표: Accuracy, AUC, Brier, Log Loss, ECE, 이닝별 지표; 테스트는 학습 후 한 번만 평가
- 출력: `phase1d_output.zip` (데이터, 오류·감사 기록, 모델, 예측, 지표, 그림)

**주의:** API 수집에 많은 요청이 필요합니다. Google Drive 체크포인트 옵션을 제공하므로 접속이 끊어져도 수집 결과를 재사용할 수 있습니다. 네트워크 오류가 남은 상태에서는 모델 학습이 실행되지 않도록 구성했습니다. 이 파일은 실제 실행 결과가 아니라 재현 가능한 분석 코드입니다.

In [ ]:
!pip -q install pandas numpy requests scikit-learn tensorflow matplotlib tqdm

In [ ]:
import io, os, re, json, zipfile, random, time, math
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, requests, matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss, log_loss
import tensorflow as tf
from google.colab import files
SEED=2026
os.environ['PYTHONHASHSEED']=str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
print('Tensorflow', tf.__version__)

## 1. 기존 Phase 1C의 분할 정보 업로드

`phase1c_output.zip`을 업로드합니다. `game_partitions.csv`를 읽어 동일한 경기와 동일한 기간 분할을 유지합니다.

In [ ]:
uploaded=files.upload()
assert len(uploaded)==1 and next(iter(uploaded)).lower().endswith('.zip'), 'phase1c_output.zip 파일 하나만 업로드하세요.'
filename=next(iter(uploaded))
with zipfile.ZipFile(io.BytesIO(uploaded[filename])) as z:
    partitions=pd.read_csv(z.open('game_partitions.csv'))
partitions['gamePk']=partitions['gamePk'].astype(int)
partitions['date']=pd.to_datetime(partitions['date']).dt.strftime('%Y-%m-%d')
assert len(partitions)==2430 and partitions.gamePk.nunique()==2430
assert set(partitions.split)=={'train','validation','test'}
assert set(partitions.target.dropna().unique())=={0,1}
print(partitions.groupby('split').agg(n=('gamePk','size'),from_date=('date','min'),to_date=('date','max')))
assert partitions.loc[partitions.split=='train','date'].max() < partitions.loc[partitions.split=='validation','date'].min()
assert partitions.loc[partitions.split=='validation','date'].max() < partitions.loc[partitions.split=='test','date'].min()

## 2. 수집 체크포인트

기본값은 Google Drive에 게임별 원본 중 **필요한 최소 요약 정보만** 저장합니다. Drive 연결을 원치 않으면 `USE_DRIVE=False`로 설정하세요. 데이터 오류 재현에 중요하므로 `phase1d_cache.jsonl`을 보존해 주세요.

In [ ]:
USE_DRIVE=True   # 선택: Drive를 이용하지 않으려면 False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTBASE=Path('/content/drive/MyDrive/MLB_Phase1D')
else:
    OUTBASE=Path('/content/MLB_Phase1D')
OUTBASE.mkdir(parents=True, exist_ok=True)
CACHE=OUTBASE/'phase1d_cache.jsonl'
ERROR_LOG=OUTBASE/'phase1d_api_errors.jsonl'
print('Cache:',CACHE)

## 3. MLB feed/live 다운로드 및 반이닝별 상태 생성

`allPlays[].about`에 나타나는 반이닝 목록을 사용해 실제로 시작된 반이닝을 식별합니다. 스냅샷을 만들 때는 **그 반이닝 이전에 완료된 반이닝**의 득점/안타만 더합니다. `linescore`의 최종 합계가 게임의 최종 결과와 일치하는지도 검사합니다.

API 요청 실패는 건너뛰되 기록하고, 모든 경기의 수집이 끝나기 전에는 학습하지 않습니다. 동일 `gamePk`에 대한 캐시가 있으면 재요청하지 않습니다.

In [ ]:
SESSION=requests.Session()
SESSION.headers.update({'User-Agent':'academic-research-mlb-phase1d/1.0','Accept':'application/json'})
def get_feed(gid,retries=5):
    url=f'https://statsapi.mlb.com/api/v1.1/game/{int(gid)}/feed/live'
    last=None
    for trial in range(retries):
        try:
            r=SESSION.get(url,timeout=(15,45))
            if r.status_code in (429,500,502,503,504):
                time.sleep(min(2**trial+1,20));continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last=repr(e); time.sleep(min(2**trial+1,20))
    raise RuntimeError(f'gamePk {gid}: API 실패 {last}')

def prepare_game(gid,feed):
    games=feed.get('gameData',{}).get('teams',{})
    live=feed.get('liveData',{})
    score=live.get('linescore',{})
    innings=score.get('innings') or []
    allplays=live.get('plays',{}).get('allPlays') or []
    if not innings or not allplays: raise ValueError('missing innings or allPlays')
    # 실제 플레이가 한 번이라도 기록된 반이닝만 생성
    appeared=set()
    for p in allplays:
        a=p.get('about') or {}
        num=a.get('inning'); half=str(a.get('halfInning','')).lower()
        if num and half in ('top','bottom'): appeared.add((int(num),half))
    if not appeared: raise ValueError('no started half-innings')
    completed={}
    for inn in innings:
        n=inn.get('num')
        if n is None:continue
        for half, side in [('top','away'),('bottom','home')]:
            data=inn.get(side) or {}
            if (int(n),half) not in appeared:continue
            runs=data.get('runs');hits=data.get('hits')
            if runs is None or hits is None:raise ValueError(f'missing runs/hits {n} {half}')
            completed[(int(n),half)]={'runs':int(runs),'hits':int(hits)}
    missing=appeared-set(completed)
    if missing:raise ValueError(f'missing half summary: {sorted(missing)[:3]}')
    seq=sorted(appeared,key=lambda x:(x[0],0 if x[1]=='top' else 1))
    hr=ar=hh=ah=0;rows=[]
    # 스냅샷은 각 반이닝의 첫 투구 전 시점. 마지막 반이닝 종료 상태는 절대 만들지 않음.
    for n,half in seq:
        rows.append({'gamePk':int(gid),'inning':n,'is_bottom':int(half=='bottom'),
                     'score_diff':hr-ar,'home_runs':hr,'away_runs':ar,
                     'home_hits':hh,'away_hits':ah,'abs_score_diff':abs(hr-ar),
                     'snapshot':'pre_half_inning'})
        v=completed[(n,half)]
        if half=='top': ar+=v['runs'];ah+=v['hits']
        else: hr+=v['runs'];hh+=v['hits']
    # 검증: 모든 반이닝을 합산한 점수가 linescore의 최종 점수와 일치해야 함
    finalh=(score.get('teams') or {}).get('home',{}).get('runs')
    finala=(score.get('teams') or {}).get('away',{}).get('runs')
    if finalh is None or finala is None: raise ValueError('missing final runs')
    if (hr,ar)!=(int(finalh),int(finala)):
        raise ValueError(f'run checksum mismatch: summed {(hr,ar)} vs final {(finalh,finala)}')
    if int(finalh)==int(finala):raise ValueError('tie final score')
    return {'gamePk':int(gid),'home_final':int(finalh),'away_final':int(finala),
            'home_win':int(int(finalh)>int(finala)),'snapshots':rows,
            'n_started_halves':len(seq),'n_extras':sum(n>9 for n,_ in seq)}

cache={};bad_cache=[]
if CACHE.exists():
    with CACHE.open() as f:
        for line in f:
            if not line.strip():continue
            try:
                obj=json.loads(line);cache[int(obj['gamePk'])]=obj
            except Exception:bad_cache.append(line[:100])
print('기존 캐시',len(cache),'파싱 오류',len(bad_cache))
errors=[];targets=dict(zip(partitions.gamePk,partitions.target))
for ix,gid in enumerate(tqdm(partitions.gamePk.tolist(),desc='MLB feed audit')):
    if int(gid) in cache:continue
    try:
        obj=prepare_game(gid,get_feed(gid))
        if int(obj['home_win'])!=int(targets[gid]):raise ValueError('target mismatch vs phase1c game_partitions')
        cache[int(gid)]=obj
        with CACHE.open('a') as f:f.write(json.dumps(obj,ensure_ascii=False)+'\n')
    except Exception as e:
        errors.append({'gamePk':int(gid),'error':str(e)})
        with ERROR_LOG.open('a') as f:f.write(json.dumps(errors[-1],ensure_ascii=False)+'\n')
    if ix%100==0 and ix>0:print('progress',ix,'cached',len(cache),'errors',len(errors))
print('Cached:',len(cache),'New errors:',len(errors))
if errors:display(pd.DataFrame(errors).head(20))

## 4. 감사(Audit) 및 학습 데이터 구축

모든 경기 검증을 통과해야 이후 셀을 실행합니다. API 구조가 일부 경기에서 다르게 기록되어 오류가 나면 오류 내용과 캐시를 먼저 확인하며, 오류가 난 경기를 임의로 제외하지 않습니다.

In [ ]:
missing=set(map(int,partitions.gamePk))-set(cache)
assert not missing, f'검증 실패/누락 gamePk: {len(missing)}건. 오류를 먼저 확인하세요. 예: {sorted(missing)[:10]}'
frames=[];bad=[]
for g in partitions.itertuples(index=False):
    o=cache[int(g.gamePk)]
    if o['home_win']!=g.target:bad.append(int(g.gamePk))
    for r in o['snapshots']:
        frames.append({**r,'date':g.date,'split':g.split,'target':int(g.target)})
assert not bad, f'Home-win target mismatch: {bad[:10]}'
df=pd.DataFrame(frames).sort_values(['date','gamePk','inning','is_bottom']).reset_index(drop=True)
assert len(df)>30000
assert df.groupby('gamePk').split.nunique().max()==1
assert df.groupby('gamePk').target.nunique().max()==1
assert len(df.gamePk.unique())==2430
assert set(df.snapshot)=={'pre_half_inning'}
FEATURES=['inning','is_bottom','score_diff','home_runs','away_runs','home_hits','away_hits','abs_score_diff']
print('스냅샷',len(df),'총 게임',df.gamePk.nunique())
print(df.groupby('split').agg(games=('gamePk','nunique'),snapshots=('gamePk','size')))
print('이닝별 표본 수');display(df.groupby(['split','inning']).size().unstack(0,fill_value=0))
assert (df[FEATURES].notnull().all()).all()
# 최종 결과 혼입 검사용: 매 경기 마지막 스냅샷의 점수와 최종 점수가 같을 수 있어도
# 마지막 반이닝 시작 전이기 때문에 해당 반이닝 진행 후 점수를 입력한 것은 아닙니다.
assert (df.loc[df.is_bottom==0,'inning']>=1).all()


## 5. 모델 학습 — Train/Validation만으로 선택

학습은 기존 논문의 64→32→16 구조를 유지합니다. 표본이 경기 안에서 반복된다는 점을 반영해 경기별 관측치 가중치를 동일하게 하고, 조기 종료는 독립적인 **8월 검증셋**에서 결정합니다. 테스트(9월)는 최종 모델 선택 후에만 평가합니다.

In [ ]:
train=df[df.split=='train'].copy();val=df[df.split=='validation'].copy();test=df[df.split=='test'].copy()
scaler=StandardScaler().fit(train[FEATURES].to_numpy(float))
Xtr=scaler.transform(train[FEATURES].to_numpy(float));Xv=scaler.transform(val[FEATURES].to_numpy(float));Xte=scaler.transform(test[FEATURES].to_numpy(float))
ytr=train.target.to_numpy(int);yv=val.target.to_numpy(int);yte=test.target.to_numpy(int)
# 한 경기가 긴 경우 행 수가 많아도 경기당 가중치 합은 동일하도록
wtr=1./train.groupby('gamePk').gamePk.transform('size').to_numpy(float)
wtr=wtr/np.mean(wtr)
model=tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(FEATURES),)),
    tf.keras.layers.Dense(64,activation='relu'),tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32,activation='relu'),tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16,activation='relu'),
    tf.keras.layers.Dense(1,activation='sigmoid')
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),loss='binary_crossentropy',metrics=['accuracy'])
cb=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=12,restore_best_weights=True)]
history=model.fit(Xtr,ytr,sample_weight=wtr,validation_data=(Xv,yv),epochs=200,batch_size=128,verbose=0,callbacks=cb)
print('DNN epochs',len(history.history['loss']),'best',np.argmin(history.history['val_loss'])+1)
logreg=LogisticRegression(max_iter=2000,random_state=SEED).fit(Xtr,ytr,sample_weight=wtr)
# 경험적 승리 기대값: 이닝·공수·점수차(-5..+5) 셀의 학습셋 경험 승률을 Laplace smoothing
train_exp=train.copy();train_exp['score_bin']=train_exp.score_diff.clip(-5,5)
prior=float((train_exp.target.to_numpy()*wtr).sum()/wtr.sum())
counts=(train_exp.assign(w=wtr,wy=wtr*train_exp.target.to_numpy())
         .groupby(['inning','is_bottom','score_bin']).agg(n=('w','sum'),wins=('wy','sum')))
def empirical(d,alpha=30.):
    keys=list(zip(d.inning.astype(int),d.is_bottom.astype(int),d.score_diff.clip(-5,5).astype(int)))
    return np.array([(float(counts.loc[k,'wins'])+alpha*prior)/(float(counts.loc[k,'n'])+alpha)
                     if k in counts.index else prior for k in keys])
# 현재 앞선 팀이 이길 것이라는 규칙. 동점이면 학습 홈팀 승률을 반환(확률은 0/1/학습 prior)
def lead_rule(d):
    s=d.score_diff.to_numpy();return np.where(s>0,1.,np.where(s<0,0.,prior))


## 6. 최종 테스트 평가, 이닝별 성능, 보정

Brier/Log Loss는 실제 확률의 정확성에 관한 지표입니다. 0/1의 경직된 규칙 기반 예측은 Log Loss가 매우 커질 수 있으므로, 해당 기준선은 주로 정확도/Brier를 중심으로 해석합니다. ECE는 10개 등간격 구간으로 계산합니다.

In [ ]:
def ece(y,p,bins=10):
    edges=np.linspace(0,1,bins+1); ids=np.minimum(np.digitize(p,edges[1:-1],right=True),bins-1)
    total=0.
    for k in range(bins):
        mask=(ids==k)
        if mask.any():total+=mask.mean()*abs(np.mean(y[mask])-np.mean(p[mask]))
    return float(total)
def scores(y,p):
    pp=np.asarray(p,float);yp=(pp>=0.5).astype(int)
    return {'n':len(y),'accuracy':float(accuracy_score(y,yp)),
            'auc':float(roc_auc_score(y,pp)) if len(set(y))==2 else np.nan,
            'brier':float(brier_score_loss(y,pp)),
            'logloss':float(log_loss(y,np.clip(pp,1e-12,1-1e-12),labels=[0,1])),
            'ece10':ece(y,pp)}
preds={'DNN':model.predict(Xte,verbose=0).reshape(-1),
       'Logistic Regression':logreg.predict_proba(Xte)[:,1],
       'Empirical Win Expectancy':empirical(test),
       'Leading-Team Rule':lead_rule(test)}
metrics=pd.DataFrame([{'model':k,**scores(yte,p)} for k,p in preds.items()])
display(metrics)
test_out=test.copy()
for name,p in preds.items():test_out['prob_'+name.lower().replace(' ','_').replace('-','_')]=p
by_inning=[]
for inn,d in test_out.groupby('inning'):
    idx=(test.inning.to_numpy()==inn)
    for name,p in preds.items():by_inning.append({'inning':int(inn),'model':name,**scores(yte[idx],p[idx])})
by_inning=pd.DataFrame(by_inning)
display(by_inning[by_inning.inning<=9].pivot(index='inning',columns='model',values='accuracy').round(3))
# 원고에 사용하는 전체 정확도는 스냅샷 기준임을 명시
print('평가 단위: 관측 시점별(pre-half-inning) 스냅샷. 9월 게임은 train/validation에 사용하지 않음.')


## 7. 결과 시각화 및 재현 패키지 저장

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4))
for name in preds:
    sub=by_inning[(by_inning.model==name)&(by_inning.inning<=9)]
    axs[0].plot(sub.inning,sub.accuracy,marker='o',label=name)
axs[0].set(xlabel='Inning',ylabel='Accuracy',title='Held-out September test by inning',ylim=(0,1));axs[0].legend(fontsize=8)
from sklearn.calibration import calibration_curve
for name in ['DNN','Logistic Regression','Empirical Win Expectancy']:
    x,y=calibration_curve(yte,preds[name],n_bins=10,strategy='uniform')
    axs[1].plot(y,x,marker='o',label=name)
axs[1].plot([0,1],[0,1],linestyle='--',color='gray')
axs[1].set(xlabel='Mean predicted probability',ylabel='Observed win rate',title='Calibration (September test)',xlim=(0,1),ylim=(0,1));axs[1].legend(fontsize=8)
plt.tight_layout()
OUT=Path('/content/phase1d_output');OUT.mkdir(exist_ok=True)
fig.savefig(OUT/'ingame_phase1d_figures.png',dpi=170);plt.show()
df.to_csv(OUT/'ingame_snapshots_2430.csv',index=False)
metrics.to_csv(OUT/'ingame_test_metrics.csv',index=False)
by_inning.to_csv(OUT/'ingame_test_by_inning.csv',index=False)
test_out.to_csv(OUT/'ingame_test_predictions.csv',index=False)
partitions.to_csv(OUT/'game_partitions.csv',index=False)
model.save(OUT/'ingame_model.keras')
export={'season':2025,'features':FEATURES,'architecture':[8,64,32,16,1],
        'scaler':{'mean':scaler.mean_.tolist(),'scale':scaler.scale_.tolist()},
        'layers':[{'W':w.tolist(),'b':b.tolist()} for w,b in [layer.get_weights() for layer in model.layers if isinstance(layer,tf.keras.layers.Dense)]]}
(OUT/'ingame_model_export.json').write_text(json.dumps(export,ensure_ascii=False))
report={'season':2025,'unique_games':int(df.gamePk.nunique()),'snapshots':len(df),
        'sampling':'start of actually played half-inning (before current half runs/hits)',
        'train_val_test_game_counts':df.groupby('split').gamePk.nunique().to_dict(),
        'train_val_test_snapshot_counts':df.groupby('split').size().to_dict(),
        'n_extra_inning_games':int(df.loc[df.inning>9,'gamePk'].nunique()),
        'innings_counts':{str(k):int(v) for k,v in df.inning.value_counts().sort_index().items()},
        'best_epoch':int(np.argmin(history.history['val_loss'])+1),
        'epochs_run':len(history.history['loss']),'n_errors':len(errors),
        'features':FEATURES,'data_assumptions':['game final results used only as training labels, not snapshot features',
                                               'score and hit tallies are pre-current-half',
                                               'starting-out/base state excluded by design',
                                               'season 2025 only; chronological split from phase1c']}
(OUT/'ingame_audit.json').write_text(json.dumps(report,indent=2,ensure_ascii=False))
(OUT/'api_errors_current_run.json').write_text(json.dumps(errors,ensure_ascii=False,indent=2))
with zipfile.ZipFile('/content/phase1d_output.zip','w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in OUT.iterdir():z.write(p,arcname=p.name)
print('OUTPUT /content/phase1d_output.zip')
files.download('/content/phase1d_output.zip')